In [16]:
import os
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models, regularizers
import matplotlib.pyplot as plt
import numpy as np
from tensorflow.keras.preprocessing import image
import kagglehub

In [26]:
path = kagglehub.dataset_download("deepcontractor/is-that-santa-image-classification")
path = os.path.join(path, "is that santa")
print("Path to dataset files:", path)


Path to dataset files: /home/inzenfenix/.cache/kagglehub/datasets/deepcontractor/is-that-santa-image-classification/versions/5/is that santa


In [27]:
TRAIN_DIR = os.path.join(path, 'train')
VAL_DIR = os.path.join(path, 'test')

print('Directorio de trabajo:', os.getcwd())
print('TRAIN_DIR existe:', os.path.exists(TRAIN_DIR))
print('VAL_DIR existe:', os.path.exists(VAL_DIR))

train_datagen = ImageDataGenerator(
    rescale=1./255, rotation_range=40, width_shift_range=0.2,
    height_shift_range=0.2, shear_range=0.2, zoom_range=0.2,
    horizontal_flip=True, fill_mode='nearest'
)
val_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(TRAIN_DIR, target_size=(224, 224), batch_size=32, class_mode='binary')
val_generator = val_datagen.flow_from_directory(VAL_DIR, target_size=(224, 224), batch_size=32, class_mode='binary')
print("Mapeo de clases:", train_generator.class_indices)


Directorio de trabajo: /home/inzenfenix/Documents/GitHub/framework-ia-UDD/Detector_santa
TRAIN_DIR existe: True
VAL_DIR existe: True
Found 614 images belonging to 2 classes.
Found 616 images belonging to 2 classes.
Mapeo de clases: {'not-a-santa': 0, 'santa': 1}


In [19]:
model = models.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(150, 150, 3)),
    layers.MaxPooling2D(2, 2),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D(2, 2),
    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.MaxPooling2D(2, 2),
    layers.Flatten(),
    layers.Dense(512, activation='relu', kernel_regularizer=regularizers.l2(0.001)),
    layers.Dropout(0.5),
    layers.Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

history = model.fit(train_generator, epochs=20, validation_data=val_generator)
model.save('detector_santa_model.h5')

/home/inzenfenix/.pyenv/versions/3.12.9/envs/tf-env/lib/python3.12/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 18s 875ms/step - accuracy: 0.5521 - loss: 1.5191 - val_accuracy: 0.6218 - val_loss: 0.9723
Epoch 2/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 18s 900ms/step - accuracy: 0.7915 - loss: 0.8304 - val_accuracy: 0.8782 - val_loss: 0.6052
Epoch 3/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 18s 892ms/step - accuracy: 0.8860 - loss: 0.6051 - val_accuracy: 0.8620 - val_loss: 0.5043
Epoch 4/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 18s 923ms/step - accuracy: 0.8681 - loss: 0.5106 - val_accuracy: 0.8474 - val_loss: 0.5891
Epoch 5/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 18s 926ms/step - accuracy: 0.8583 - loss: 0.5317 - val_accuracy: 0.8831 - val_loss: 0.4179
Epoch 6/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 19s 933ms/step - accuracy: 0.8844 - loss: 0.4396 - val_accuracy: 0.9075 - val_loss: 0.3541
Epoch 7/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 19s 932ms/step - accuracy: 0.8893 - loss: 0.3912 - val_accuracy: 0.9075 - val_loss: 0.3287
Epoch 8/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 19s 933ms/step - accuracy: 0.8925 - loss: 0.3779 - val_accu

In [23]:
def clasificar_imagen(ruta_imagen):
    img = image.load_img(ruta_imagen, target_size=(150, 150))
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0) / 255.0

    probabilidad = model.predict(img_array)[0][0]
    if probabilidad > 0.5:
        print(f"¡Es SANTA Claus! 🎅 ({probabilidad*100:.2f}%)")
    else:
        print(f"No es Santa ❌ ({(1 - probabilidad)*100:.2f}%)")

# Para probar una foto, solo cambia el nombre aquí y corre esta celda:
clasificar_imagen('./Datos/test/santa/103.Santa.jpg')

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
¡Es SANTA Claus! 🎅 (78.72%)
